# 📊 Mutual Fund Analytics — Exploratory Data Analysis

**Day 1 EDA Notebook**  
Explore the live NAV data fetched from MFAPI across 6 mutual fund schemes.

---
### Contents
1. Load & inspect the dataset  
2. NAV trend visualization  
3. Returns analysis (1M / 3M / 1Y)  
4. Risk metrics (volatility, drawdown, Sharpe)  
5. Fund comparison  
6. Key takeaways


In [ ]:
# ── Imports ─────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 11})

print('All libraries loaded successfully.')

## 1. Load & Inspect Dataset

In [ ]:
# Load the processed (enriched) dataset
DATA_PATH = Path('../data/processed/nav_cleaned.csv')

df = pd.read_csv(DATA_PATH, parse_dates=['date'])
df['nav'] = pd.to_numeric(df['nav'], errors='coerce')
df = df.sort_values(['scheme_name', 'date']).reset_index(drop=True)

print(f'Shape       : {df.shape}')
print(f'Date range  : {df["date"].min().date()}  to  {df["date"].max().date()}')
print(f'Funds       : {df["scheme_name"].nunique()}')
print(f'Null values : {df.isnull().sum().sum()}')
df.head()

In [ ]:
# Per-fund record counts
summary = df.groupby('scheme_name').agg(
    Records=('nav', 'count'),
    First_Date=('date', 'min'),
    Last_Date=('date', 'max'),
    Latest_NAV=('nav', 'last'),
).reset_index()
summary['Fund'] = summary['scheme_name'].str[:50]
summary[['Fund','Records','First_Date','Last_Date','Latest_NAV']]

## 2. NAV Trend Visualization

In [ ]:
# ── Normalised NAV comparison (Base = 100) ───────────────────────
fig, ax = plt.subplots(figsize=(14, 6))

colors = ['#3b82f6','#10b981','#8b5cf6','#f59e0b','#ef4444','#0d9488']

for i, (fund, fdf) in enumerate(df.groupby('scheme_name')):
    fdf = fdf.sort_values('date')
    base = fdf['nav'].iloc[0]
    norm_nav = fdf['nav'] / base * 100
    label = fund[:40] + '...' if len(fund) > 40 else fund
    ax.plot(fdf['date'], norm_nav, label=label, color=colors[i % len(colors)],
            linewidth=1.8, alpha=0.9)

ax.axhline(100, linestyle='--', color='#94a3b8', linewidth=0.8, alpha=0.6)
ax.set_title('NAV Performance (Normalised to 100 at Start)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Date')
ax.set_ylabel('Indexed Value (Base = 100)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9, framealpha=0.9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Last 3 years only ────────────────────────────────────────────
cutoff = df['date'].max() - pd.Timedelta(days=3*365)
df_3y  = df[df['date'] >= cutoff]

fig = px.line(
    df_3y, x='date', y='nav', color='scheme_name',
    title='NAV History — Last 3 Years (Interactive)',
    labels={'date': 'Date', 'nav': 'NAV (Rs.)', 'scheme_name': 'Fund'},
    template='plotly_white',
)
fig.update_traces(line_width=2)
fig.update_layout(legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1))
fig.show()

## 3. Returns Analysis

In [ ]:
# ── Compute multi-period returns ─────────────────────────────────
def period_return(fdf, days):
    cutoff = fdf['date'].max() - pd.Timedelta(days=days)
    sub = fdf[fdf['date'] >= cutoff].sort_values('date')
    if len(sub) < 2 or sub['nav'].iloc[0] == 0:
        return None
    return (sub['nav'].iloc[-1] / sub['nav'].iloc[0] - 1) * 100

rows = []
for fund, fdf in df.groupby('scheme_name'):
    fdf = fdf.sort_values('date')
    total = (fdf['nav'].iloc[-1] / fdf['nav'].iloc[0] - 1) * 100
    years = (fdf['date'].max() - fdf['date'].min()).days / 365
    cagr  = ((fdf['nav'].iloc[-1] / fdf['nav'].iloc[0]) ** (1/years) - 1) * 100 if years > 0 else None
    rows.append({
        'Fund'         : fund[:45],
        '1M Return %'  : period_return(fdf, 30),
        '3M Return %'  : period_return(fdf, 90),
        '1Y Return %'  : period_return(fdf, 365),
        'Total Return %': total,
        'CAGR %'       : cagr,
    })

returns_df = pd.DataFrame(rows).set_index('Fund')
returns_df.round(2)

In [ ]:
# ── Returns bar chart ────────────────────────────────────────────
ret_plot = returns_df[['1Y Return %','Total Return %']].dropna().sort_values('1Y Return %')
ret_plot.plot(kind='barh', figsize=(12, 5), color=['#3b82f6', '#10b981'], alpha=0.85)
plt.title('Fund Returns: 1-Year vs Total', fontsize=13, fontweight='bold')
plt.xlabel('Return (%)')
plt.axvline(0, color='black', linewidth=0.8)
plt.legend()
plt.tight_layout()
plt.show()

## 4. Risk Metrics

In [ ]:
# ── Volatility, Sharpe, Max Drawdown ─────────────────────────────
risk_rows = []
for fund, fdf in df.groupby('scheme_name'):
    fdf = fdf.sort_values('date')
    daily = fdf['nav'].pct_change().clip(-0.5, 0.5).dropna()
    vol   = daily.std() * np.sqrt(252) * 100
    rf    = 0.065 / 252
    sharpe = ((daily - rf).mean() / daily.std() * np.sqrt(252)) if daily.std() > 0 else 0
    roll_max = fdf['nav'].expanding().max()
    max_dd   = ((fdf['nav'] - roll_max) / roll_max * 100).min()
    risk_rows.append({
        'Fund'        : fund[:45],
        'Volatility %': round(vol, 2),
        'Sharpe Ratio': round(sharpe, 2),
        'Max Drawdown %': round(max_dd, 2),
    })

risk_df = pd.DataFrame(risk_rows).set_index('Fund')
risk_df

In [ ]:
# ── Correlation heatmap of daily returns ─────────────────────────
pivot = df.pivot_table(index='date', columns='scheme_name', values='daily_return')
pivot.columns = [c[:30] for c in pivot.columns]

corr = pivot.corr()
fig, ax = plt.subplots(figsize=(10, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=-1, vmax=1, center=0, ax=ax,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation of Daily Returns Between Funds', fontsize=13, fontweight='bold', pad=12)
plt.xticks(rotation=30, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

## 5. Fund Comparison — Risk vs Return

In [ ]:
# ── Risk-Return scatter ──────────────────────────────────────────
# Merge returns_df and risk_df
compare = returns_df[['1Y Return %']].join(risk_df[['Volatility %','Sharpe Ratio']]).dropna()
compare = compare[compare['Volatility %'] < 100]   # exclude extreme outliers

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#3b82f6','#10b981','#8b5cf6','#f59e0b','#ef4444','#0d9488']

for i, (fund, row) in enumerate(compare.iterrows()):
    ax.scatter(row['Volatility %'], row['1Y Return %'],
               s=180, color=colors[i % len(colors)], zorder=5, edgecolors='white', linewidth=1.5)
    ax.annotate(fund[:30], (row['Volatility %'], row['1Y Return %']),
                textcoords='offset points', xytext=(8, 4), fontsize=8.5)

ax.axhline(0, color='#94a3b8', linewidth=0.8, linestyle='--')
ax.set_xlabel('Annualised Volatility (%)', fontsize=11)
ax.set_ylabel('1-Year Return (%)', fontsize=11)
ax.set_title('Risk vs Return Map (1-Year)', fontsize=13, fontweight='bold')
ax.annotate('Ideal: High Return, Low Risk', xy=(0.02, 0.96),
            xycoords='axes fraction', fontsize=9, color='#64748b', style='italic')
plt.tight_layout()
plt.show()

## 6. Key Takeaways

In [ ]:
# ── Auto-generate summary ────────────────────────────────────────
full = returns_df.join(risk_df).dropna()
full = full[full['Volatility %'] < 100]   # remove outliers

best_ret   = full['Total Return %'].idxmax()
best_sharpe = full['Sharpe Ratio'].idxmax()
most_stable = full['Volatility %'].idxmin()
worst_dd    = full['Max Drawdown %'].idxmin()

print('=== KEY FINDINGS ===')
print(f'Best Total Return : {best_ret}  ({full.loc[best_ret,"Total Return %"]:+.1f}%)')
print(f'Best Sharpe Ratio : {best_sharpe}  ({full.loc[best_sharpe,"Sharpe Ratio"]:.2f})')
print(f'Most Stable Fund  : {most_stable}  (vol={full.loc[most_stable,"Volatility %"]:.1f}%)')
print(f'Worst Drawdown    : {worst_dd}  ({full.loc[worst_dd,"Max Drawdown %"]:.1f}%)')
print()
print('Full scorecard:')
full.round(2)